In [ ]:
import streamlit as st
import os
from openai import OpenAI
from sympy import content, re
# from anthropic import Anthropic
import pdfplumber
from io import StringIO
from dotenv import load_dotenv
import pandas as pd
import os
from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader,
    UnstructuredWordDocumentLoader,
    UnstructuredFileLoader
)
from langchain_groq import Groq


/home/ahmed/miniconda3/envs/ai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load environment variables
load_dotenv(override=True)

True

In [25]:

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("GROQ_API_KEY")
google_api_key = os.getenv("GROQ_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")
deepseek_api_key = os.getenv("GROQ_API_KEY")

openai = OpenAI()

In [26]:
def load_and_split_resume(file_path: str):
    """
    Loads a resume file and splits it into text chunks using LangChain.

    Args:
        file_path (str): Path to the resume file (.txt, .pdf, .docx, etc.)
        chunk_size (int): Maximum characters per chunk.
        chunk_overlap (int): Overlap between chunks to preserve context.

    Returns:
        List[str]: List of split text chunks.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    ext = os.path.splitext(file_path)[1].lower()

    # Select the appropriate loader
    if ext == ".txt":
        loader = TextLoader(file_path, encoding="utf-8")
    elif ext == ".pdf":
        loader = PyPDFLoader(file_path)
    elif ext in [".docx", ".doc"]:
        loader = UnstructuredWordDocumentLoader(file_path)
    else:
        # Fallback for other common formats
        loader = UnstructuredFileLoader(file_path)

    # Load the file as LangChain documents
    documents = loader.load()

   
    return documents
    # return [doc.page_content for doc in split_docs]


def extract_text(file):
    """
    Extract text from uploaded file.
    Supports PDF and TXT files.
    Returns extracted text as string.
    """

    try:
        # ----------- PDF FILE -----------
        if file.name.lower().endswith(".pdf"):
            texts = []

            with pdfplumber.open(file) as pdf:
                for page in pdf.pages:
                    text = page.extract_text()
                    if text:
                        texts.append(text)

            return "\n".join(texts)

        # ----------- TEXT FILE -----------
        elif file.name.lower().endswith(".txt"):
            content = file.read()

            # If content is bytes, decode it
            if isinstance(content, bytes):
                content = content.decode("utf-8", errors="ignore")

            return content

        # ----------- UNSUPPORTED FORMAT -----------
        else:
            return "Unsupported file format. Please upload PDF or TXT file."

    except Exception as e:
        return f"Error extracting text: {str(e)}"

def extract_candidate_name(resume_text):
    prompt = f"""
You are an AI assistant specialized in resume analysis.

Your task is to get full name of the candidate from the resume.

Resume:
{resume_text}

Respond with only the candidate's full name.
""" 
    try:
        from openai import OpenAI
        openai = OpenAI()
        response = openai.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a professional resume evaluator."},
                {"role": "user", "content": prompt}
            ]
        )
        content = response.choices[0].message.content
        
        return content.strip()

    except Exception as e:
        return "Unknown"


# Function to build the prompt for LLMs
def build_prompt(resume_text, jd_text):
    prompt = f"""
    You are an expert Technical Recruiter. Your task is to provide a high-precision match score between a candidate's Resume and a Job Description (JD).

    ### Evaluation Criteria:
    1. Hard Skills Match (Tools, languages, frameworks).
    2. Experience Level (Years of experience and seniority).
    3. Industry Fit (Relevant projects and domain knowledge).

    ### Scoring Scale:
    - 0-30%: Little to no relevance.
    - 31-60%: Some matching skills, but lacks core requirements.
    - 61-90%: Strong match; fits most requirements.
    - 91-100%: Perfect fit; exceeds requirements.

    ### Data:
    RESUME:
    {resume_text}

    JOB DESCRIPTION:
    {jd_text}

    ### Output:
    Respond ONLY with the match percentage as a single integer (e.g., 85). Do not include any text, symbols, or explanations.
    """
    return prompt.strip()


In [ ]:
import re
import openai
from langchain_groq import ChatGroq


def get_openai_match(prompt):
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a professional resume evaluator."},
            {"role": "user", "content": prompt}
        ]
    )
    content = response.choices[0].message.content # Added this line
    match = re.search(r'\d+', content)
    percentage = int(match.group()) if match else 0
    return min(percentage, 100)


# 1. Correct Initialization
# The error "field required [type=missing]" usually means 'model' was not found.
def get_groq_client(model_name="llama-3.3-70b-versatile"):
    return ChatGroq(
        groq_api_key=groq_api_key, 
        model_name=model_name,
        temperature=0
    )

def extract_score(ai_response):
    """Helper to pull the integer out of the response object"""
    content = ai_response.content
    match = re.search(r'\d+', content)
    return min(int(match.group()), 100) if match else 0

def get_groq_match(prompt):
    # Mapping "OpenAI-level" quality to Groq's best model
    llm = get_groq_client("llama-3.3-70b-versatile")
    response = llm.invoke(prompt)
    return extract_score(response)

def get_anthropic_match(prompt):
    # Mapping "Anthropic-level" speed/logic to Llama 3.1
    llm = get_groq_client("llama-3.1-8b-instant")
    response = llm.invoke(prompt)
    return extract_score(response)

def get_google_match(prompt):
    # Mapping "Gemini-level" to Mixtral
    llm = get_groq_client("openai/gpt-oss-120b")
    response = llm.invoke(prompt)
    return extract_score(response)

def get_deepseek_match(prompt):
    # Using Llama 3.1 70b as a DeepSeek alternative on Groq
    llm = get_groq_client("groq/compound-mini")
    response = llm.invoke(prompt)
    return extract_score(response)

In [42]:
resume_file = 'files/Tanvir_Ahmed_CV.pdf'
jd_file = 'files/job.txt'

In [44]:
import os

print(os.path.exists(resume_file))
print(os.path.getsize(resume_file))

True
35625


In [45]:
from langchain_community.document_loaders import PyPDFLoader

# Load Resume
loader = PyPDFLoader(resume_file)
resume_docs = loader.load()

resume_text = "\n".join([doc.page_content for doc in resume_docs])

# Load Job Description
with open(jd_file, "r", encoding="utf-8") as f:
    jd_text = f.read()

print(resume_text[:500])
print("*"*50)
print(jd_text[:500])

TANVIR AHMED
Machine Learning Engineer
📍 Dhaka, Bangladesh
📧 
tanvirahmed754575@gmail.com
📞 +880 1744279706
🔗 
GitHub
🔗 
LinkedIn
PROFESSIONAL SUMMARY
Machine Learning Engineer specializing in building production-ready AI systems.
Experienced in developing end-to-end machine learning pipelines — from data
preprocessing and feature engineering to deployment and monitoring. Strong
background in Computer Vision, Generative AI, and MLOps with hands-on
experience in PyTorch, TensorFlow, FastAPI, Dock
**************************************************
Job Title: Machine Learning Engineer (Generative AI & MLOps)

Location: Remote / Dhaka, Bangladesh

Experience: 2–4 Years
Role Overview We are seeking a versatile Machine Learning Engineer to join our AI team. You will be responsible for designing, building, and deploying production-ready AI systems. The ideal candidate has a blend of expertise in Deep Learning (Computer Vision), Generative AI (LLMs/RAG), and a "DevOps mindset" for managing the

In [46]:
jd_text

'Job Title: Machine Learning Engineer (Generative AI & MLOps)\n\nLocation: Remote / Dhaka, Bangladesh\n\nExperience: 2–4 Years\nRole Overview We are seeking a versatile Machine Learning Engineer to join our AI team. You will be responsible for designing, building, and deploying production-ready AI systems. The ideal candidate has a blend of expertise in Deep Learning (Computer Vision), Generative AI (LLMs/RAG), and a "DevOps mindset" for managing the full ML lifecycle. You won\'t just train models; you will build the pipelines that keep them running.\nKey Responsibilities\n\n    End-to-End Development: Design and implement scalable ML pipelines, from data ingestion and feature engineering to model deployment and monitoring.\n\n    Generative AI & Agents: Develop advanced RAG (Retrieval-Augmented Generation) systems and multi-agent workflows using frameworks like LangChain, CrewAI, and Pinecone.\n\n    Computer Vision: Build and optimize object detection and image segmentation models us

In [68]:
candidate_name = extract_candidate_name(resume_text)
prompt = build_prompt(resume_text, jd_text)

# Get match percentages from all models
scores = {
    "OpenAI GPT-4o Mini": get_openai_match(prompt),
    "Anthropic Claude": get_anthropic_match(prompt),
    "Google Gemini": get_google_match(prompt),
    "Groq": get_groq_match(prompt),
    "DeepSeek": get_deepseek_match(prompt),
}


In [69]:
scores

{'OpenAI GPT-4o Mini': 95,
 'Anthropic Claude': 95,
 'Google Gemini': 94,
 'Groq': 92,
 'DeepSeek': 94}

In [70]:
# Calculate average score
average_score = round(sum(scores.values()) / len(scores), 2)

# Sort scores in descending order
sorted_scores = sorted(scores.items(), reverse=False)

In [71]:
candidate_name

'Tanvir Ahmed'

In [72]:
prompt 

'You are an expert Technical Recruiter. Your task is to provide a high-precision match score between a candidate\'s Resume and a Job Description (JD).\n\n    ### Evaluation Criteria:\n    1. Hard Skills Match (Tools, languages, frameworks).\n    2. Experience Level (Years of experience and seniority).\n    3. Industry Fit (Relevant projects and domain knowledge).\n\n    ### Scoring Scale:\n    - 0-30%: Little to no relevance.\n    - 31-60%: Some matching skills, but lacks core requirements.\n    - 61-90%: Strong match; fits most requirements.\n    - 91-100%: Perfect fit; exceeds requirements.\n\n    ### Data:\n    RESUME:\n    TANVIR AHMED\nMachine Learning Engineer\n📍 Dhaka, Bangladesh\n📧 \ntanvirahmed754575@gmail.com\n📞 +880 1744279706\n🔗 \nGitHub\n🔗 \nLinkedIn\nPROFESSIONAL SUMMARY\nMachine Learning Engineer specializing in building production-ready AI systems.\nExperienced in developing end-to-end machine learning pipelines — from data\npreprocessing and feature engineering to depl

In [73]:
average_score

94.0

In [74]:
scores_df = pd.DataFrame(scores.items(), columns=["Model", "Match Percentage"])
scores_df = scores_df.sort_values(by="Match Percentage", ascending=False).reset_index(drop=True)
scores_df

,Model,Match Percentage
0,OpenAI GPT-4o Mini,95
1,Anthropic Claude,95
2,Google Gemini,94
3,DeepSeek,94
4,Groq,92


In [75]:
sorted_scores

[('Anthropic Claude', 95),
 ('DeepSeek', 94),
 ('Google Gemini', 94),
 ('Groq', 92),
 ('OpenAI GPT-4o Mini', 95)]